In [0]:
# Imports and configuration
import json
import time
import requests
import subprocess
from jinja2 import Environment, FileSystemLoader
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import mlflow

In [0]:
# Load utils
UTILS_DIR = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/utils"

exec(open(f"{UTILS_DIR}/config.py").read())
exec(open(f"{UTILS_DIR}/states.py").read())
exec(open(f"{UTILS_DIR}/llm_utils.py").read())

print("✅ Utils loaded")

In [0]:
# Read Silver products
df_products = spark.table(SILVER_PRODUCTS)
print(f"Products to process: {df_products.count()}")
display(df_products.select("product_id", "product_description"))

In [0]:
def extract_product(product_description: str) -> dict:
    
    system_prompt = load_prompt(
        PROMPTS_DIR, PROMPT_EXTRACTION, role="system",
        allowed_subcategories=ALLOWED_SUBCATEGORIES
    )
    user_prompt = load_prompt(
        PROMPTS_DIR, PROMPT_EXTRACTION, role="user",
        product_description=product_description
    )
    
    result = call_llm(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        api_key      = LLM_API_KEY,
        api_url      = LLM_API_URL,
        model        = LLM_MODEL,
        output_model = ProductExtraction  # ← valida e retenta se inválido
    )
    # validated_output já é um objeto ProductExtraction tipado
    return {
        "extracted"    : result["validated_output"],
        "input_tokens" : result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "latency"      : result["latency_seconds"]
    }

In [0]:
# Process all products — metrics go ONLY to MLflow
mlflow.set_experiment(MLFLOW_EXPERIMENT)

products = df_products.select("product_id", "product_description").collect()
results  = []

with mlflow.start_run(run_name=f"extraction_{PROMPT_VERSION}"):

    # Log prompt templates as MLflow artifacts — shows structure not values
    extraction_template_raw = load_prompt_template_raw(PROMPTS_DIR, PROMPT_EXTRACTION)
    mlflow.log_text(extraction_template_raw, PROMPT_EXTRACTION)

    # Log run parameters
    mlflow.log_param("prompt_version",  PROMPT_VERSION)
    mlflow.log_param("model",           LLM_MODEL)
    mlflow.log_param("provider",        LLM_PROVIDER)
    mlflow.log_param("total_products",  len(products))

    success_count    = 0
    failure_count    = 0
    total_input_tok  = 0
    total_output_tok = 0
    total_latency    = 0.0

    for row in products:
        product_id          = row["product_id"]
        product_description = row["product_description"]

        try:
            result    = extract_product(product_description)
            time.sleep(5)  # 5 seconds between each product
            extracted = result["extracted"]

            # Accumulate metrics for MLflow
            total_input_tok  += result["input_tokens"]
            total_output_tok += result["output_tokens"]
            total_latency    += result["latency"]
            success_count    += 1

            results.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : extracted.name,
                "extracted_brand"       : extracted.brand,
                "extracted_sub_category": extracted.sub_category,
                "llm_status"            : "success",
            })

        except Exception as e:
            failure_count += 1
            results.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : None,
                "extracted_brand"       : None,
                "extracted_sub_category": None,
                "llm_status"            : f"failed: {str(e)}",
            })

        time.sleep(3)

    # Log all metrics to MLflow — single source of truth for observability
    mlflow.log_metric("success_count",       success_count)
    mlflow.log_metric("failure_count",       failure_count)
    mlflow.log_metric("success_rate",        round(success_count / len(products), 3))
    mlflow.log_metric("total_input_tokens",  total_input_tok)
    mlflow.log_metric("total_output_tokens", total_output_tok)
    mlflow.log_metric("total_tokens",        total_input_tok + total_output_tok)
    mlflow.log_metric("avg_latency_seconds", round(total_latency / len(products), 3))

print(f"✅ Extraction complete: {success_count} success, {failure_count} failed")

In [0]:
# Save to Delta

df_extracted = spark.createDataFrame(results)

(df_extracted.write
    .format("delta")
    .mode("overwrite")
    # .option("mergeSchema", "true")
    .option("overwriteSchema", "true")
    .saveAsTable(LLM_EXTRACTED))

print(f"✅ Written: {LLM_EXTRACTED}")


In [0]:
display(df_extracted)